# Лабораторная работа №3 — RAG-система для PDF

**Студент:** Фам Минь Хиеу · **Группа:** МПИ-25-1-2 · **Москва, 2026**

Реализованы задания 1–7 из методички. Исходный PDF — Ouyang et al. (2022), «Training language models to follow instructions with human feedback» (`LLM instruct (2).pdf`, 68 страниц). Все таблицы пересчитываются при запуске. **Предыдущие outputs удалены, так как изменены промпты и методика проверки.**

**Среда:** LM Studio Local Server (`http://localhost:1234/v1`), модель генерации Qwen3-8B и **отдельная** модель плотных embeddings (например, `nomic-embed-text-v1.5`). Имя embedding-модели должно совпадать с ID в LM Studio. На вкладке Developer включите сервер; обе модели должны быть доступны серверу. В отличие от примера методички с `OllamaEmbeddings`, здесь плотные embeddings получаются через совместимый API LM Studio; сама методика сравнения не меняется.

**Важно:** ноутбук рассчитан на локальный Jupyter/VS Code на том же компьютере, где запущен LM Studio. В обычном облачном Google Colab `localhost:1234` недоступен.
**Повторное выполнение:** файл PDF должен называться `LLM instruct (2).pdf` и находиться в рабочей директории. API LM Studio доступен локально на `http://localhost:1234/v1`; Google Colab хранит результаты, но для запуска в облаке потребуется отдельная настройка доступа к локальному серверу. Результаты и оценки ниже являются результатами ранее выполненного запуска и не пересчитывались при оформлении итогового комплекта.


In [1]:
%pip -q install pymupdf scikit-learn pandas numpy requests tabulate openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os, re, json, time, requests
import fitz, numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

PDF_PATH = os.environ.get('PDF_PATH', 'LLM instruct (2).pdf')
LM_STUDIO_URL = os.environ.get('LM_STUDIO_URL', 'http://localhost:1234/v1').rstrip('/')
EMBED_MODEL = os.environ.get('EMBED_MODEL', 'text-embedding-nomic-embed-text-v1.5')
LLM_MODEL = os.environ.get('LLM_MODEL', 'qwen/qwen3-8b')
K = 3
TEMPERATURE = 0
print('PDF:', PDF_PATH, '| LM Studio:', LM_STUDIO_URL)
if not Path(PDF_PATH).exists():
    raise FileNotFoundError('Загрузите PDF в рабочую папку (Colab: Files → Upload), затем укажите PDF_PATH')
pages = [p.get_text('text') for p in fitz.open(PDF_PATH)]
print('Страниц:',len(pages),'символов:',sum(map(len,pages)))
assert sum(map(len,pages)) > 6000

def split_pdf(chunk_size=1000, chunk_overlap=10):
    assert 0 <= chunk_overlap < chunk_size
    chunks=[]
    for page_no, page in enumerate(pages, 1):
        start=0
        while start < len(page):
            fragment=page[start:start+chunk_size]
            if fragment.strip():
                chunks.append(dict(id=len(chunks)+1, page=page_no, source=Path(PDF_PATH).name,
                                   start_index=start, text=fragment))
            if start+chunk_size >= len(page): break
            start += chunk_size-chunk_overlap
    return chunks

def api_post(path, payload):
    response=requests.post(f'{LM_STUDIO_URL}/{path}', json=payload, timeout=600)
    if not response.ok:
        raise RuntimeError(f'LM Studio {path}: HTTP {response.status_code}; {response.text[:500]}')
    return response.json()

def dense_embed(texts):
    # Batch to limit server memory use, maintain input order.
    vectors=[]
    for start in range(0,len(texts),16):
        block=texts[start:start+16]
        data=api_post('embeddings', {'model':EMBED_MODEL,'input':block})['data']
        vectors.extend([np.asarray(item['embedding'],dtype=np.float32)
                        for item in sorted(data,key=lambda item:item['index'])])
    matrix=np.asarray(vectors,dtype=np.float32)
    return matrix / np.maximum(np.linalg.norm(matrix,axis=1,keepdims=True),1e-12)

def lm_generate(prompt):
    data=api_post('chat/completions', {'model':LLM_MODEL,
      'messages':[{'role':'user','content':prompt + '\n/no_think'}],
      'temperature':TEMPERATURE,'seed':42,'max_tokens':1024,'stream':False})
    return (data['choices'][0]['message'].get('content') or '').strip()

try:
    r=requests.get(f'{LM_STUDIO_URL}/models',timeout=10)
    r.raise_for_status()
    available=[item['id'] for item in r.json().get('data',[])]
    print('Доступные модели LM Studio:', available)
    if LLM_MODEL not in available or EMBED_MODEL not in available:
        raise ValueError('Укажите реальные ID загруженных моделей в LLM_MODEL / EMBED_MODEL. '
                         f'Доступно: {available}')
    assert dense_embed(['embedding test']).shape[0] == 1
    assert lm_generate('Respond with the word READY.').strip()
except Exception as e:
    raise RuntimeError('Проверьте вкладку Developer → Start Server и загрузку Qwen3-8B '
                       'и отдельной embedding-модели в LM Studio. '
                       f'Диагностика: {e}') from e

PDF: LLM instruct (2).pdf | LM Studio: http://localhost:1234/v1
Страниц: 68 символов: 181855
Доступные модели LM Studio: ['qwen/qwen3-8b', 'text-embedding-nomic-embed-text-v1.5']


## Методика и эталонные вопросы

Страницы ниже — **порядковые номера PDF (с 1)**, а не напечатанные номера страниц статьи. Эталоны соответствуют Ouyang et al. (2022).

In [3]:
QA = [
 ('What is the title of the paper?', 'Training language models to follow instructions with human feedback.', 1, 'Training language models to follow instructions'),
 ('What name is given to the resulting models?', 'InstructGPT.', 1, 'We call the resulting models InstructGPT'),
 ('Which smaller InstructGPT model is preferred over 175B GPT-3?', 'The 1.3B parameter InstructGPT model.', 1, '1.3B parameter InstructGPT'),
 ('What are the three main training steps?', 'Supervised fine-tuning, reward modeling, and PPO reinforcement learning.', 3, 'three steps'),
 ('How many contractors were hired to label the data?', '40 contractors.', 7, '40 contractors'),
 ('What is learned from ranked model responses?', 'A reward model that predicts preferred responses.', 8, 'reward model'),
 ('Which training variant mixes in pretraining data?', 'PPO-ptx.', 9, 'PPO-ptx'),
 ('How often were 175B InstructGPT outputs preferred over 175B GPT-3?', '85 ± 3% of the time.', 3, '85 ± 3%'),
 ('Which benchmark is used to study factual truthfulness?', 'TruthfulQA.', 10, 'TruthfulQA'),
 ('Does the paper discuss limitations?', 'Yes, it discusses limitations of the approach.', 19, 'limitations'),
]
# Автоматически сверяем, что формулировки эталонов действительно присутствуют.
for q, answer, page, needle in QA:
    assert page <= len(pages) and needle.casefold() in pages[page-1].casefold(), (q,page,needle)
qa_df=pd.DataFrame(QA, columns=['Вопрос','Эталонный ответ','Страница PDF','Опорная цитата'])
display(qa_df)

def dense_index(chunks): return {'chunks':chunks,'vectors':dense_embed([c['text'] for c in chunks])}
def dense_search(idx, query, k=3):
    qv=dense_embed([query])
    scores=(idx['vectors'] @ qv[0])
    return [(idx['chunks'][int(i)],float(scores[i])) for i in np.argsort(-scores)[:k]]
def bow_index(chunks):
    vectorizer=CountVectorizer(binary=True,lowercase=True,token_pattern=r'(?u)\b\w\w+\b')
    matrix=vectorizer.fit_transform([c['text'] for c in chunks])
    return {'chunks':chunks,'vectorizer':vectorizer,'matrix':matrix}
def bow_search(idx, query, k=3):
    q=idx['vectorizer'].transform([query]); values=cosine_similarity(q,idx['matrix']).ravel()
    if q.nnz==0: return []
    return [(idx['chunks'][int(i)],float(values[i])) for i in np.argsort(-values)[:k]]
def evidence_found(hits, item):
    """Automatic *proxy*, not a final expert label: anchor evidence in a retrieved chunk."""
    clean=lambda v: ' '.join(v.casefold().split())
    return any(c['page']==item[2] and clean(item[3]) in clean(c['text']) for c,_ in hits)

def inspect_retrieval(question, item, hits):
    """A printable checklist for manually judging top-3 answer sufficiency."""
    return {'Вопрос':question,'Эталон':item[1], 'Страница PDF':item[2],
            'Авто-якорь':int(evidence_found(hits,item)),
            'Top-3 страницы':[c['page'] for c,_ in hits],
            'Top-3 ID':[c['id'] for c,_ in hits],
            'Достаточно контекста 0/1':'ПРОВЕРИТЬ ВРУЧНУЮ'}
def references(hits):
    return '\n\n'.join(f"[Фрагмент {c['id']}, стр. {c['page']}; {c['source']}]\n{c['text']}" for c,_ in hits)
BASE_PROMPT = 'Answer the question using this context.\nContext:\n{context}\nQuestion: {question}\nAnswer:'
STRICT_PROMPT = ("You must answer ONLY from the supplied PDF excerpts. "
 "Give a substantive answer with evidence citations exactly like [Фрагмент 12, стр. 4]. "
 "Every factual claim must have a supporting citation from the supplied excerpts. "
 "A citation alone is not a valid answer. Never invent a fragment ID or page. "
 "If the context is insufficient to answer, say exactly: 'Not enough information in the document.' "
 "Do not treat conversation history as evidence.\n"
 "Context:\n{context}\nQuestion: {question}\nAnswer:")

def is_refusal(response):
    import re
    patterns=[r'not enough information', r'insufficient information',
              r'not (?:specified|provided|mentioned|available) in (?:the )?(?:document|context)',
              r'no information (?:in|about)',r'cannot (?:determine|answer)',
              r'недостаточно информации',r'нет информации',r'не указано',r'не содержится']
    return any(re.search(p,response.casefold()) for p in patterns)

def answer(question, hits, prompt=STRICT_PROMPT):
    return lm_generate(prompt.format(context=references(hits),question=question))
def show_hits(q,hits):
    print('Исходный вопрос:',q,'\nПоисковый запрос:',q)
    for c,s in hits: print(f"  ID={c['id']} стр={c['page']} start={c['start_index']} score={s:.3f}: {c['text'][:160]!r}")

,Вопрос,Эталонный ответ,Страница PDF,Опорная цитата
0,What is the title of the paper?,Training language models to follow instruction...,1,Training language models to follow instructions
1,What name is given to the resulting models?,InstructGPT.,1,We call the resulting models InstructGPT
2,Which smaller InstructGPT model is preferred o...,The 1.3B parameter InstructGPT model.,1,1.3B parameter InstructGPT
3,What are the three main training steps?,"Supervised fine-tuning, reward modeling, and P...",3,three steps
4,How many contractors were hired to label the d...,40 contractors.,7,40 contractors
5,What is learned from ranked model responses?,A reward model that predicts preferred responses.,8,reward model
6,Which training variant mixes in pretraining data?,PPO-ptx.,9,PPO-ptx
7,How often were 175B InstructGPT outputs prefer...,85 ± 3% of the time.,3,85 ± 3%
8,Which benchmark is used to study factual truth...,TruthfulQA.,10,TruthfulQA
9,Does the paper discuss limitations?,"Yes, it discusses limitations of the approach.",19,limitations


## Задание 1 — первые пять фрагментов против полного индекса

In [4]:
chunks=split_pdf(1000,10)
assert len(chunks)>5
early=[x for x in QA if any(x[3].casefold() in c['text'].casefold() and c['page']==x[2] for c in chunks[:5])][:3]
late=[x for x in QA if x not in early and not any(x[3].casefold() in c['text'].casefold() and c['page']==x[2] for c in chunks[:5])][:3]
assert len(early)==3 and len(late)==3, 'Нужно подобрать 3 ранних и 3 поздних вопроса'
idx5=dense_index(chunks[:5]); idxall=dense_index(chunks)
task1=[]
for name,idx in [('Первые 5',idx5),('Полный',idxall)]:
    for group,questions in [('начало',early),('дальше',late)]:
        for item in questions:
            hits=dense_search(idx,item[0],K); response=answer(item[0],hits)
            show_hits(item[0],hits); print('Ответ:',response)
            task1.append(dict(Индекс=name,Группа=group,Вопрос=item[0],Авто_якорь=int(evidence_found(hits,item)),Ответ=response))
t1=pd.DataFrame(task1)
display(t1.groupby(['Индекс','Группа'])['Авто_якорь'].agg(['sum','count']))
display(t1.groupby('Индекс')['Авто_якорь'].mean().rename('Hit Rate@3'))
print('Оценку ответа LLM 0–2 выставьте вручную по эталонам и цитатам.')
print("ВНИМАНИЕ: Авто-якорь — предварительная метрика; проверьте top-3 вручную до финального Hit Rate@3.")


Исходный вопрос: What is the title of the paper? 
Поисковый запрос: What is the title of the paper?
  ID=3 стр=1 start=1980 score=0.475: 'aking up facts, generating biased or toxic text, or simply not following\nuser instructions (Bender et al., 2021; Bommasani et al., 2021; Kenton et al., 2021; We'
  ID=4 стр=2 start=0 score=0.451: '1.3B\n6B\n175B\nModel size\n0.2\n0.4\n0.6\nWin rate against SFT 175B\nModel\nPPO-ptx\nPPO\nSFT\nGPT (prompted)\nGPT\nFigure 1: Human evaluations of various models on our API '
  ID=2 стр=1 start=990 score=0.398: 'rvised\nlearning. We then collect a dataset of rankings of model outputs, which we use to\nfurther ﬁne-tune this supervised model using reinforcement learning fro'
Ответ: Not enough information in the document.
Исходный вопрос: What name is given to the resulting models? 
Поисковый запрос: What name is given to the resulting models?
  ID=2 стр=1 start=990 score=0.583: 'rvised\nlearning. We then collect a dataset of rankings of model outputs, which

sum  count
Индекс   Группа            
Первые 5 дальше    0      3
         начало    2      3
Полный   дальше    1      3
         начало    0      3

Индекс
Первые 5    0.333333
Полный      0.166667
Name: Hit Rate@3, dtype: float64

Оценку ответа LLM 0–2 выставьте вручную по эталонам и цитатам.
ВНИМАНИЕ: Авто-якорь — предварительная метрика; проверьте top-3 вручную до финального Hit Rate@3.


## Задание 2 — длина фрагмента (overlap=10, k=3)

In [5]:
t2=[]
for size in [500,1000,2000]:
    cs=split_pdf(size,10); ix=dense_index(cs)
    outcomes=[int(evidence_found(dense_search(ix,item[0],K),item)) for item in QA]
    t2.append({'chunk_size':size,'Число фрагментов':len(cs),'Успехов из 10':sum(outcomes),'Hit Rate@3':np.mean(outcomes)})
display(pd.DataFrame(t2))
print("Hit Rate@3 выше — автоматический proxy на основании точного якоря; дополните ручной проверкой достаточности контекста.")


,chunk_size,Число фрагментов,Успехов из 10,Hit Rate@3
0,500,398,4,0.4
1,1000,215,4,0.4
2,2000,127,2,0.2


Hit Rate@3 выше — автоматический proxy на основании точного якоря; дополните ручной проверкой достаточности контекста.


## Задание 3 — перекрытие (chunk_size=1000, k=3)

In [6]:
# Отбираем вопросы по страницам, у которых опорный текст ближе всего к границам chunk_size.
def boundary_distance(item):
    text=pages[item[2]-1]; pos=text.casefold().find(item[3].casefold())
    return min(pos%1000,1000-(pos%1000)) if pos>=0 else 9999
boundary_q=sorted(QA,key=boundary_distance)[:5]
print('Пограничные вопросы и расстояние:',[(x[0],boundary_distance(x)) for x in boundary_q])
t3=[]
for overlap in [0,100,200]:
    cs=split_pdf(1000,overlap); ix=dense_index(cs)
    good=0; duplicates=0
    for item in boundary_q:
        hits=dense_search(ix,item[0],K)
        good+=evidence_found(hits,item)
        # Приближение повторов: коэффициент Жаккара множеств слов >=0.8.
        sets=[set(re.findall(r'\w+',c['text'].lower())) for c,_ in hits]
        duplicated=any(len(a&b)/max(len(a|b),1)>=.8 for i,a in enumerate(sets) for b in sets[i+1:])
        duplicates+=duplicated
    t3.append({'overlap':overlap,'Число фрагментов':len(cs),'Успехов из 5':good,'Вопросов с повторами':duplicates})
display(pd.DataFrame(t3))
print("Для задания 3 проверьте пограничные случаи по тексту: технический критерий дубля Жаккар >= 0.8.")


Пограничные вопросы и расстояние: [('What is the title of the paper?', 0), ('Does the paper discuss limitations?', 4), ('What is learned from ranked model responses?', 31), ('What are the three main training steps?', 37), ('How often were 175B InstructGPT outputs preferred over 175B GPT-3?', 73)]


,overlap,Число фрагментов,Успехов из 5,Вопросов с повторами
0,0,215,3,0
1,100,227,2,0
2,200,242,2,0


Для задания 3 проверьте пограничные случаи по тексту: технический критерий дубля Жаккар >= 0.8.


## Задание 4 — бинарный мешок слов против LM Studio dense embeddings

In [7]:
PARAPHRASES=[
 'How many phases make up the human-feedback alignment pipeline?',
 'What was the size of the team hired for human annotations?',
 'What model estimates which response people would favor?',
 'Which algorithmic variant retains a language-modeling data mixture?',
 'Does the study acknowledge shortcomings of its method?',
]
exact=[QA[i] for i in [3,4,5,6,9]]
testset=[('точный',x[0],x) for x in exact]+[('перефраз',q,x) for q,x in zip(PARAPHRASES,exact)]
bow=bow_index(chunks); dense=idxall; t4=[]
for method,idx,search in [('BoW',bow,bow_search),('LM Studio embeddings',dense,dense_search)]:
    for group,q,item in testset:
        hits=search(idx,q,K); success=int(evidence_found(hits,item))
        t4.append({'Метод':method,'Группа':group,'Вопрос':q,'Успех':success,
                   'Найденные ID':[c['id'] for c,_ in hits]})
t4=pd.DataFrame(t4)
display(t4.groupby(['Метод','Группа'])['Успех'].agg(['sum','mean']))
display(t4)
print("Результаты — автоматическая оценка якоря. Для окончательной оценки выполните ручную проверку содержания фрагментов.")


sum  mean
Метод                Группа             
BoW                  перефраз    0   0.0
                     точный      1   0.2
LM Studio embeddings перефраз    2   0.4
                     точный      2   0.4

,Метод,Группа,Вопрос,Успех,Найденные ID
0,BoW,точный,What are the three main training steps?,1,"[8, 14, 7]"
1,BoW,точный,How many contractors were hired to label the d...,0,"[133, 34, 126]"
2,BoW,точный,What is learned from ranked model responses?,0,"[36, 127, 136]"
3,BoW,точный,Which training variant mixes in pretraining data?,0,"[185, 152, 19]"
4,BoW,точный,Does the paper discuss limitations?,0,"[83, 180, 1]"
5,BoW,перефраз,How many phases make up the human-feedback ali...,0,"[13, 76, 14]"
6,BoW,перефраз,What was the size of the team hired for human ...,0,"[175, 149, 189]"
7,BoW,перефраз,What model estimates which response people wou...,0,"[136, 80, 141]"
8,BoW,перефраз,Which algorithmic variant retains a language-m...,0,"[19, 34, 17]"
9,BoW,перефраз,Does the study acknowledge shortcomings of its...,0,"[60, 106, 159]"


Результаты — автоматическая оценка якоря. Для окончательной оценки выполните ручную проверку содержания фрагментов.


## Задание 5 — цитирования и проверка фактов

In [8]:
citation_re=re.compile(r'\[Фрагмент\s+(\d+),\s*стр\.\s*(\d+)\]')
t5=[]
for item in QA[:5]:
    hits=dense_search(idxall,item[0],K)
    out=answer(item[0],hits)
    cites=[(int(a),int(b)) for a,b in citation_re.findall(out)]
    src={c['id']:c for c,_ in hits}
    exists=bool(cites) and all(i in src for i,p in cites)
    pages_ok=exists and all(src[i]['page']==p for i,p in cites)
    citation_only=bool(cites) and not citation_re.sub('',out).strip(' \n.,;:-')
    t5.append({'Вопрос':item[0], 'Эталон':item[1],
        'Ответ и ссылки':out,'Ссылки существуют':exists,
        'Страницы верны':pages_ok, 'Только ссылка без ответа':citation_only,
        'Смысл утверждения подтвержден':'ПРОВЕРИТЬ ВРУЧНУЮ',
        'Top-3 доказательства':references(hits)})
display(pd.DataFrame(t5).drop(columns=['Top-3 доказательства']))
print('Для каждой ссылки необходимо вручную проверить, что сам текст подтверждает утверждение.')

,Вопрос,Эталон,Ответ и ссылки,Ссылки существуют,Страницы верны,Только ссылка без ответа,Смысл утверждения подтвержден
0,What is the title of the paper?,Training language models to follow instruction...,Not enough information in the document.,False,False,False,ПРОВЕРИТЬ ВРУЧНУЮ
1,What name is given to the resulting models?,InstructGPT.,"The resulting models are referred to as ""instr...",False,False,False,ПРОВЕРИТЬ ВРУЧНУЮ
2,Which smaller InstructGPT model is preferred o...,The 1.3B parameter InstructGPT model.,The 1.3B PPO-ptx model is preferred over 175B ...,True,True,False,ПРОВЕРИТЬ ВРУЧНУЮ
3,What are the three main training steps?,"Supervised fine-tuning, reward modeling, and P...",The three main training steps are: (1) pretrai...,False,False,False,ПРОВЕРИТЬ ВРУЧНУЮ
4,How many contractors were hired to label the d...,40 contractors.,"[Фрагмент 77, стр. 19; LLM instruct (2).pdf] T...",False,False,False,ПРОВЕРИТЬ ВРУЧНУЮ


Для каждой ссылки необходимо вручную проверить, что сам текст подтверждает утверждение.


## Задание 6 — ответы и обоснованные отказы

In [9]:
NO_ANSWER=[
 'What was the exact IP address of the training server?',
 'What was the annual salary of the lead annotator?',
 'What was the personal phone number of the first author?',
 'What was the exact electricity bill for the training run?',
 'What password protected the annotation database?',
]
assert all(not any(q.casefold() in page.casefold() for page in pages) for q in NO_ANSWER)
items=[(x[0],1,x) for x in QA[:5]]+[(q,0,None) for q in NO_ANSWER]
precomputed={q:dense_search(idxall,q,K) for q,_,_ in items}
t6=[]
for prompt_name,prompt in [('Исходный',BASE_PROMPT),('Строгий',STRICT_PROMPT)]:
    for q,has_answer,item in items:
        out=answer(q,precomputed[q],prompt)
        refused=is_refusal(out)
        t6.append({'Промпт':prompt_name,'Вопрос':q,'Ответ присутствует':has_answer,
                   'Ответ модели':out,'Отказ':refused,
                   'Контекст содержит эталон':evidence_found(precomputed[q],item) if item else False})
t6=pd.DataFrame(t6)
print('Исходный промпт:',BASE_PROMPT,'\nСтрогий промпт:',STRICT_PROMPT)
display(t6.groupby('Промпт').apply(lambda a:pd.Series({
 'Корректные отказы из 5':int(a.loc[a['Ответ присутствует']==0,'Отказ'].sum()),
 'Ответы без отказа для вопросов без ответа':int((~a.loc[a['Ответ присутствует']==0,'Отказ']).sum()),
 'Средний балл ответов 0–2':'ПРОВЕРИТЬ ВРУЧНУЮ'}),include_groups=False))
display(t6)
print("Контроль: различайте правильный отказ для отсутствующего факта и необоснованный отказ при достаточном контексте.")


Исходный промпт: Answer the question using this context.
Context:
{context}
Question: {question}
Answer: 
Строгий промпт: You must answer ONLY from the supplied PDF excerpts. Give a substantive answer with evidence citations exactly like [Фрагмент 12, стр. 4]. Every factual claim must have a supporting citation from the supplied excerpts. A citation alone is not a valid answer. Never invent a fragment ID or page. If the context is insufficient to answer, say exactly: 'Not enough information in the document.' Do not treat conversation history as evidence.
Context:
{context}
Question: {question}
Answer:


,Корректные отказы из 5,Ответы без отказа для вопросов без ответа,Средний балл ответов 0–2
Промпт,,,
Исходный,0,5,ПРОВЕРИТЬ ВРУЧНУЮ
Строгий,5,0,ПРОВЕРИТЬ ВРУЧНУЮ


,Промпт,Вопрос,Ответ присутствует,Ответ модели,Отказ,Контекст содержит эталон
0,Исходный,What is the title of the paper?,1,The title of the paper is not explicitly provi...,False,False
1,Исходный,What name is given to the resulting models?,1,"The resulting models are referred to as ""RLHF ...",False,False
2,Исходный,Which smaller InstructGPT model is preferred o...,1,The 1.3B PPO-ptx model is preferred over 175B ...,False,False
3,Исходный,What are the three main training steps?,1,The three main training steps are:\n\n1. **Pre...,False,False
4,Исходный,How many contractors were hired to label the d...,1,"The context states that ""We hired about 40 con...",False,False
5,Исходный,What was the exact IP address of the training ...,0,The context provided does not mention the exac...,False,False
6,Исходный,What was the annual salary of the lead annotator?,0,The context provided does not mention the annu...,False,False
7,Исходный,What was the personal phone number of the firs...,0,The context provided does not include any info...,False,False
8,Исходный,What was the exact electricity bill for the tr...,0,The context provided does not mention anything...,False,False
9,Исходный,What password protected the annotation database?,0,The provided context does not contain any info...,False,False


Контроль: различайте правильный отказ для отсутствующего факта и необоснованный отказ при достаточном контексте.


## Задание 7 — три диалога, переформулирование, сессии

In [10]:
sessions={}
def clear_session(session_id): sessions.pop(session_id,None)
def rewrite(question,history):
    if not history: return question
    hist='\n'.join(f"Вопрос: {q}\nОтвет: {a}" for q,a in history[-3:])
    return lm_generate('Перепиши последний вопрос в самостоятельный поисковый запрос. '
      'Используй историю лишь для раскрытия местоимений. НЕ отвечай на вопрос, '
      'НЕ добавляй новые сведения. Дай только запрос.\nИстория:\n'+hist+'\nПоследний вопрос: '+question)
def chat(session_id,question,mode='Б'):
    history=sessions.setdefault(session_id,[])
    query=rewrite(question,history) if mode=='Б' else question
    hits=dense_search(idxall,query,K)
    hist='\n'.join(f'Вопрос: {q}\nОтвет: {a}' for q,a in history[-3:])
    response=answer('История (не доказательство):\n'+hist+'\nТекущий вопрос: '+question,hits)
    history.append((question,response))
    show_hits(query,hits); print('Исходная реплика:',question,'\nОтвет:',response)
    return {'Сессия':session_id,'Тип':mode,'Исходная реплика':question,
            'Запрос для поиска':query,'ID фрагментов':[c['id'] for c,_ in hits],
            'Ответ':response,'Ответ 0–2':'ПРОВЕРИТЬ ВРУЧНУЮ'}
DIALOGS=[
 ['What is InstructGPT?','How was it trained?','Why did the authors use human feedback?'],
 ['What is the reward model in this paper?','How was it trained?','How was it used during reinforcement learning?'],
 ['What is PPO-ptx?','Why did the authors add pretraining data?','How did its results compare?']]
t7=[]
for i,dialog in enumerate(DIALOGS,1):
    # Проверяем оба метода на одинаковой паре независимых сессий.
    for mode in ['А','Б']:
        sid=f'dialog{i}_{mode}'
        for step,q in enumerate(dialog,1):
            record=chat(sid,q,mode)
            if step>=2: t7.append({'Диалог/шаг':f'{i}/{step}',**record})
assert len(sessions['dialog1_А'])==3
assert 'unseen_session' not in sessions
clear_session('dialog1_А'); assert 'dialog1_А' not in sessions
print('Проверены изоляция сессий, повторное обращение и очистка.')
display(pd.DataFrame(t7))
print("Проверьте 12 уточнений: восстановление объекта, наличие достаточных доказательств, качество ответа 0–2.")


Исходный вопрос: What is InstructGPT? 
Поисковый запрос: What is InstructGPT?
  ID=46 стр=11 start=1980 score=0.669: 'low explicit constraints deﬁned in the instruction (e.g. “Write your\nanswer in 2 paragraphs or less.”), are less likely to fail to follow the correct instructio'
  ID=9 стр=3 start=990 score=0.655: 'uctions. Outputs from our 175B\nInstructGPT are preferred to 175B GPT-3 outputs 85 ± 3% of the time, and preferred 71 ± 4% of the\ntime to few-shot 175B GPT-3. In'
  ID=48 стр=12 start=990 score=0.638: 'on the\nInstructGPT prompt distribution. FLAN and T0 perform better than default GPT-3, and comparably\nwith a few-shot GPT-3 model placed into ‘instruction-follo'
Исходная реплика: What is InstructGPT? 
Ответ: InstructGPT refers to a series of models that have been fine-tuned to follow instructions more effectively than the original GPT-3 model. These models are designed to improve reliability, truthfulness, and adherence to explicit constraints in the instruction. Accordin

,Диалог/шаг,Сессия,Тип,Исходная реплика,Запрос для поиска,ID фрагментов,Ответ,Ответ 0–2
0,1/2,dialog1_А,А,How was it trained?,How was it trained?,"[185, 187, 189]",InstructGPT was trained using a combination of...,ПРОВЕРИТЬ ВРУЧНУЮ
1,1/3,dialog1_А,А,Why did the authors use human feedback?,Why did the authors use human feedback?,"[15, 28, 1]",The authors used human feedback to align the l...,ПРОВЕРИТЬ ВРУЧНУЮ
2,1/2,dialog1_Б,Б,How was it trained?,How was InstructGPT trained?,"[48, 9, 60]",InstructGPT was trained using a combination of...,ПРОВЕРИТЬ ВРУЧНУЮ
3,1/3,dialog1_Б,Б,Why did the authors use human feedback?,Why did the authors use human feedback in trai...,"[2, 7, 48]",The authors used human feedback to align the l...,ПРОВЕРИТЬ ВРУЧНУЮ
4,2/2,dialog2_А,А,How was it trained?,How was it trained?,"[185, 187, 189]",The reward model in this paper was trained on ...,ПРОВЕРИТЬ ВРУЧНУЮ
5,2/3,dialog2_А,А,How was it used during reinforcement learning?,How was it used during reinforcement learning?,"[81, 2, 98]",The reward model in this paper was used during...,ПРОВЕРИТЬ ВРУЧНУЮ
6,2/2,dialog2_Б,Б,How was it trained?,How was the reward model trained?,"[22, 157, 31]",The reward model in this paper was trained on ...,ПРОВЕРИТЬ ВРУЧНУЮ
7,2/3,dialog2_Б,Б,How was it used during reinforcement learning?,How was the reward model utilized in the reinf...,"[36, 22, 8]",The reward model in this paper was used during...,ПРОВЕРИТЬ ВРУЧНУЮ
8,3/2,dialog3_А,А,Why did the authors add pretraining data?,Why did the authors add pretraining data?,"[185, 178, 180]",The authors added pretraining data to address ...,ПРОВЕРИТЬ ВРУЧНУЮ
9,3/3,dialog3_А,А,How did its results compare?,How did its results compare?,"[82, 29, 32]",The results of the PPO-ptx model were compared...,ПРОВЕРИТЬ ВРУЧНУЮ


Проверьте 12 уточнений: восстановление объекта, наличие достаточных доказательств, качество ответа 0–2.


## Сводка и оформление отчёта (пункты 14–15)

После выполнения всех ячеек:

1. Сверьте каждый эталон с реальным PDF и поставьте оценки ответам LLM (0/1/2) вручную.
2. Для таблицы ссылок проверьте **каждое утверждение**, а не только корректность номера страницы.
3. Опишите минимум две ошибки поиска при сокращённом индексе, примеры потери/дублирования контекста, ошибки генерации и запросов.
4. Добавьте ссылку на доступный Colab, сохраните `.ipynb` **с outputs** (`File → Save a copy in Drive`, затем `Download .ipynb`).
5. Сформулируйте вывод только из полученных чисел: одинаковый Hit Rate — допустимый результат.

Требования 14: титульный лист; цель/задачи; PDF; Colab URL; эталоны с цитатами/страницами и метками «нет ответа»; таблицы и промпты; примеры поиска и генерации; классификация ошибок; вывод. Критерии 15: 10+10+15+10+15+10+10+15+5=100 баллов.

**Важно:** ячейки до выполнения не являются экспериментальным результатом. Данный notebook — воспроизводимая реализация и протокол; итоговая оценка требует полного прогона и ручной проверки.

**Важное ограничение:** автоматический «якорь на нужной странице» — только proxy. Итоговые Hit Rate@3, наличие подтверждения и оценки 0–2 требуют проверки содержания фрагментов вручную. Сравнение dense embeddings выполняется через LM Studio вместо предписанного в примере OllamaEmbeddings; отклонение указать в отчёте.


In [11]:
# Автоматический черновик отчёта из фактических outputs (запускается после заданий 1–7).
def table(df): return df.to_markdown(index=False)
parts=[
 '# Отчёт по лабораторной работе №3',
 '**Фам Минь Хиеу, МПИ-25-1-2. Москва, 2026.**',
 '## Цель и задачи',
 'Разработать и исследовать RAG для PDF; сравнить индексацию, chunk size, overlap, поиск, ссылки, отказы и память диалога.',
 '## Исходный PDF и воспроизводимость',
 f'Файл: {Path(PDF_PATH).name}. Страниц: {len(pages)}. Фрагментов (1000/10): {len(chunks)}.',
 f'Поиск: LM Studio dense embeddings ({EMBED_MODEL}), генерация: LM Studio ({LLM_MODEL}), температура: {TEMPERATURE}, top-k: {K}.',
 '**Ссылка на Colab:** ДОБАВИТЬ после загрузки. **Файл:** см. приложенный ipynb с outputs.',
 '## Эталонные вопросы и ответы',table(qa_df),
 '## Задание 1',table(t1[['Индекс','Группа','Вопрос','Авто_якорь','Ответ']]),
 '## Задание 2',table(pd.DataFrame(t2)),
 '## Задание 3',table(pd.DataFrame(t3)),
 '## Задание 4',table(t4),
 '## Задание 5',table(pd.DataFrame(t5)),
 '## Задание 6 — промпты',
 'Исходный: '+BASE_PROMPT+'\n\nСтрогий: '+STRICT_PROMPT,table(t6),
 '## Задание 7',table(pd.DataFrame(t7)),
 '## Анализ ошибок и выводы — ручная верификация обязательна',
 '- **Подготовка данных:** ...',
 '- **Поиск:** ... (не менее двух случаев для первых пяти фрагментов)',
 '- **Преобразование запроса:** ...',
 '- **Генерация и отказы:** ...',
 '- **Оформление ссылок и подтверждение утверждений:** ...',
 '- **Обоснование размера фрагментов, overlap и метода поиска:** ...',
 '- **Оценки ответов LLM по шкале 0–2:** ...',
 '- **Вывод о настройках по результатам эксперимента:** ...',
 'Примечание: отсутствие улучшения — допустимый результат при корректном сравнении.']
Path('Отчет_ЛР3_Фам_Минь_Хиеу_черновик.md').write_text('\n\n'.join(parts),encoding='utf-8')
print('Отчёт сохранён: Отчет_ЛР3_Фам_Минь_Хиеу_черновик.md')

Отчёт сохранён: Отчет_ЛР3_Фам_Минь_Хиеу_черновик.md


## Итоговая проверка критериев 14–15 (без повторной генерации)

Запускайте эту ячейку **после всех семи заданий** в том же Python kernel. Она использует сохранённые переменные `t1`–`t7`, не вызывает LLM повторно и создаёт Excel-файл для ручной проверки доказательств. **Не выдумывайте оценки:** введите их после сверки с PDF. Автоматическая проверка только по якорным словам не является достаточной оценкой Hit Rate@3.

Отказ может быть сформулирован разными словами. Допускаются ссылки `[Фрагмент N, стр. P]` и `[Фрагмент N, стр. P; имя PDF]`.

In [12]:
import re
from pathlib import Path

# Пересчитать только проверки по уже сохранённым ответам — вызовов LM Studio нет.
CITE_FLEX=re.compile(r'\[Фрагмент\s+(\d+),\s*стр\.\s*(\d+)(?:;[^\]]+)?\]')
REFUSAL_FLEX=[r'not enough information',r'insufficient information',
    r'not (?:explicitly )?(?:specified|provided|mentioned|available)',
    r'does not (?:mention|include|contain)',r'not (?:mentioned|included|specified) in',
    r'no information',r'недостаточно информации',r'нет информации',r'не указано']
def refusal_flex(text):
    return any(re.search(p,str(text).casefold()) for p in REFUSAL_FLEX)

# Проверяем цитаты по фактически переданным top-3; имя файла необязательно.
cit_audit=[]
for i,item in enumerate(QA[:5]):
    q=item[0]; hits=dense_search(idxall,q,K)
    response=t5[i]['Ответ и ссылки']
    cites=[(int(a),int(b)) for a,b in CITE_FLEX.findall(response)]
    available={c['id']:c for c,s in hits}
    valid=bool(cites) and all(fid in available and available[fid]['page']==page for fid,page in cites)
    substantial=bool(CITE_FLEX.sub('',response).strip(' \n.,:;-')) and not refusal_flex(response)
    cit_audit.append({'Вопрос':q,'Ответ':response,'Ссылки':str(cites),
        'ID и страница верны':valid,'Есть содержательный ответ':substantial,
        'Подтверждение факта (0/1)':'ПРОВЕРИТЬ ВРУЧНУЮ',
        'Контекст top-3':references(hits)})
cit_audit=pd.DataFrame(cit_audit)

t6_rechecked=t6.copy()
t6_rechecked['Отказ (повторная проверка)']=t6_rechecked['Ответ модели'].map(refusal_flex)
display(t6_rechecked.groupby('Промпт').apply(
    lambda d:pd.Series({'Отказы на 5 вопросах без ответа':int(d.loc[d['Ответ присутствует']==0,'Отказ (повторная проверка)'].sum()),
                        'Отказы на 5 вопросах с ответом':int(d.loc[d['Ответ присутствует']==1,'Отказ (повторная проверка)'].sum())}),
    include_groups=False))
display(cit_audit.drop(columns=['Контекст top-3']))

# Фрагменты для каждого уточнения: reconstruct from saved ID, zero model calls.
reference_idx={c['id']:c for c in chunks}
dialog_audit=pd.DataFrame(t7)
dialog_audit['Контекст top-3']=dialog_audit['ID фрагментов'].map(
    lambda ids:'\n\n'.join(f"[Фрагмент {i}, стр. {reference_idx[i]['page']}]\n{reference_idx[i]['text']}"
                          for i in ids if i in reference_idx))
dialog_audit['Объект восстановлен (0/1)']='ПРОВЕРИТЬ ВРУЧНУЮ'
dialog_audit['Достаточный контекст (0/1)']='ПРОВЕРИТЬ ВРУЧНУЮ'
dialog_audit['Баллы ответа (0/1/2)']='ПРОВЕРИТЬ ВРУЧНУЮ'

# Не заполнять цифры без семантической проверки контекста.
with pd.ExcelWriter('ЛР3_экспертная_проверка.xlsx') as writer:
    qa_df.to_excel(writer,sheet_name='Эталоны',index=False)
    pd.DataFrame(t2).to_excel(writer,sheet_name='Задание_2',index=False)
    pd.DataFrame(t3).to_excel(writer,sheet_name='Задание_3',index=False)
    t4.to_excel(writer,sheet_name='Задание_4',index=False)
    cit_audit.to_excel(writer,sheet_name='Задание_5',index=False)
    t6_rechecked.to_excel(writer,sheet_name='Задание_6',index=False)
    dialog_audit.to_excel(writer,sheet_name='Задание_7',index=False)
print('Создан файл ЛР3_экспертная_проверка.xlsx — заполните проверки по PDF.')

,Отказы на 5 вопросах без ответа,Отказы на 5 вопросах с ответом
Промпт,,
Исходный,5,1
Строгий,5,1


,Вопрос,Ответ,Ссылки,ID и страница верны,Есть содержательный ответ,Подтверждение факта (0/1)
0,What is the title of the paper?,Not enough information in the document.,[],False,False,ПРОВЕРИТЬ ВРУЧНУЮ
1,What name is given to the resulting models?,"The resulting models are referred to as ""instr...","[(76, 18)]",True,True,ПРОВЕРИТЬ ВРУЧНУЮ
2,Which smaller InstructGPT model is preferred o...,The 1.3B PPO-ptx model is preferred over 175B ...,"[(4, 2)]",True,True,ПРОВЕРИТЬ ВРУЧНУЮ
3,What are the three main training steps?,The three main training steps are: (1) pretrai...,"[(189, 58)]",True,True,ПРОВЕРИТЬ ВРУЧНУЮ
4,How many contractors were hired to label the d...,"[Фрагмент 77, стр. 19; LLM instruct (2).pdf] T...","[(77, 19)]",True,True,ПРОВЕРИТЬ ВРУЧНУЮ


Создан файл ЛР3_экспертная_проверка.xlsx — заполните проверки по PDF.


## Дополнительное исследование: улучшенный RAG (после заданий 1–7)

**Этот раздел является дополнительным. Он НЕ подменяет обязательные сравнения в заданиях 1–7.**

Цель: сравнить (1) плотный поиск, (2) гибридный плотный + BM25, (3) гибридный + reranking. Документ, исходные вопросы и `top-k=3` одинаковы. Поиск сначала получает до 20 кандидатов, после чего возвращает три фрагмента. Гибрид использует Reciprocal Rank Fusion (RRF). Для reranking используется небольшой CrossEncoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) на CPU. При отсутствии модели выполняется явно обозначенный **лексический fallback**, а не скрытая подмена метода.

Для экономии времени генерация LLM выполняется только для пяти контрольных вопросов. Таблицы `Hit Rate@3` — предварительные оценки по эталонному якорю и странице; итоговое экспертное решение требует чтения найденных фрагментов. Не делайте вывода о лучшем варианте без проверки и измерений.

In [13]:
%pip -q install rank-bm25 sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from rank_bm25 import BM25Okapi
import re, numpy as np, pandas as pd

def tokenize_en(text):
    return re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*",text.casefold())

lex_tokens=[tokenize_en(c['text']) for c in chunks]
bm25_model=BM25Okapi(lex_tokens)
by_id={c['id']:c for c in chunks}

def bm25_search(query,k=20):
    scores=bm25_model.get_scores(tokenize_en(query))
    return [(chunks[int(i)],float(scores[i])) for i in np.argsort(-scores)[:k]]

def hybrid_search(query,k=3,candidates=20,rrf_k=60):
    dense=dense_search(idxall,query,candidates)
    lexical=bm25_search(query,candidates)
    fused={}
    for ranking in (dense,lexical):
        for rank,(chunk,_) in enumerate(ranking,1):
            fid=chunk['id'];fused[fid]=fused.get(fid,0)+1/(rrf_k+rank)
    ids=sorted(fused,key=fused.get,reverse=True)[:k]
    return [(by_id[fid],float(fused[fid])) for fid in ids]

USE_CROSS_ENCODER=True
RERANKER_NAME='cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker=None
RERANK_STATUS='lexical fallback'
if USE_CROSS_ENCODER:
    try:
        from sentence_transformers import CrossEncoder
        reranker=CrossEncoder(RERANKER_NAME,device='cpu')
        RERANK_STATUS=f'CrossEncoder {RERANKER_NAME}'
    except Exception as exc:
        print('CrossEncoder unavailable; explicit lexical fallback:',str(exc)[:240])

def rerank_search(query,k=3,candidates=20):
    pool=hybrid_search(query,k=candidates,candidates=candidates)
    if reranker is not None:
        scores=np.asarray(reranker.predict([(query,c['text']) for c,_ in pool]),dtype=float).reshape(-1)
    else:
        q=set(tokenize_en(query))
        scores=np.asarray([len(q.intersection(set(tokenize_en(c['text']))))/max(len(q),1)
                           for c,_ in pool],dtype=float)
    order=np.argsort(-scores)[:k]
    return [(pool[int(i)][0],float(scores[int(i)])) for i in order]

print('Reranking implementation:',RERANK_STATUS)
for title,func in [('Dense',lambda q:dense_search(idxall,q,3)),
                   ('Hybrid',lambda q:hybrid_search(q,3)),
                   ('Hybrid + rerank',lambda q:rerank_search(q,3))]:
    res=func(QA[0][0]);print(title,[(c['id'],c['page']) for c,_ in res])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranking implementation: CrossEncoder cross-encoder/ms-marco-MiniLM-L-6-v2
Dense [(116, 27), (121, 30), (146, 39)]
Hybrid [(165, 47), (115, 27), (209, 66)]
Hybrid + rerank [(75, 18), (165, 47), (83, 20)]


### Сравнение поиска по 10 исходным вопросам

Для каждого вопроса сохраняются полный текст top-3 и ссылки на страницу. Колонка `Авто-якорь` проверяет наличие эталонной фразы на указанной странице внутри найденных фрагментов, но **не заменяет содержательную проверку**.

In [15]:
searchers={
    'Dense':lambda q:dense_search(idxall,q,K),
    'Hybrid (RRF)':lambda q:hybrid_search(q,K),
    'Hybrid + rerank':lambda q:rerank_search(q,K),
}
improve_rows=[]
for method,searcher in searchers.items():
    for item in QA:
        q=item[0];hits=searcher(q)
        improve_rows.append({'Метод':method,'Вопрос':q,'Эталон':item[1],
          'Страница эталона':item[2],'Авто-якорь':int(evidence_found(hits,item)),
          'ID top-3':[c['id'] for c,_ in hits], 'Страницы top-3':[c['page'] for c,_ in hits],
          'Текст top-3':references(hits),'Достаточный контекст 0/1':'ПРОВЕРИТЬ'})
improve_df=pd.DataFrame(improve_rows)
display(improve_df.groupby('Метод')['Авто-якорь'].agg(['sum','count','mean']))
display(improve_df.drop(columns=['Текст top-3']).head(12))

,sum,count,mean
Метод,,,
Dense,4,10,0.4
Hybrid (RRF),5,10,0.5
Hybrid + rerank,6,10,0.6


,Метод,Вопрос,Эталон,Страница эталона,Авто-якорь,ID top-3,Страницы top-3,Достаточный контекст 0/1
0,Dense,What is the title of the paper?,Training language models to follow instruction...,1,0,"[116, 121, 146]","[27, 30, 39]",ПРОВЕРИТЬ
1,Dense,What name is given to the resulting models?,InstructGPT.,1,0,"[76, 78, 57]","[18, 19, 14]",ПРОВЕРИТЬ
2,Dense,Which smaller InstructGPT model is preferred o...,The 1.3B parameter InstructGPT model.,1,0,"[44, 4, 46]","[11, 2, 11]",ПРОВЕРИТЬ
3,Dense,What are the three main training steps?,"Supervised fine-tuning, reward modeling, and P...",3,0,"[185, 189, 187]","[56, 58, 58]",ПРОВЕРИТЬ
4,Dense,How many contractors were hired to label the d...,40 contractors.,7,0,"[77, 130, 132]","[19, 34, 35]",ПРОВЕРИТЬ
5,Dense,What is learned from ranked model responses?,A reward model that predicts preferred responses.,8,1,"[31, 14, 2]","[8, 4, 1]",ПРОВЕРИТЬ
6,Dense,Which training variant mixes in pretraining data?,PPO-ptx.,9,0,"[185, 189, 187]","[56, 58, 58]",ПРОВЕРИТЬ
7,Dense,How often were 175B InstructGPT outputs prefer...,85 ± 3% of the time.,3,1,"[44, 9, 4]","[11, 3, 2]",ПРОВЕРИТЬ
8,Dense,Which benchmark is used to study factual truth...,TruthfulQA.,10,1,"[40, 139, 158]","[10, 37, 44]",ПРОВЕРИТЬ
9,Dense,Does the paper discuss limitations?,"Yes, it discusses limitations of the approach.",19,1,"[77, 91, 87]","[19, 21, 21]",ПРОВЕРИТЬ


### Генерация и проверка ссылок для пяти контрольных вопросов

Номера фрагментов ограничены контекстом. Если Qwen создаёт неверные ссылки, система фиксирует ошибку и делает **одну** повторную генерацию с напоминанием разрешённых ID. Результаты и ошибка сохраняются; проверка ссылки не означает подтверждения самого утверждения.

In [16]:
CITE_PATTERN=re.compile(r'\[Фрагмент\s+(\d+),\s*стр\.\s*(\d+)(?:;[^\]]+)?\]')
REFUSE_PATTERN=re.compile(r'not enough information|insufficient information|недостаточно информации',re.I)

def validate_references(response,hits):
    claims=CITE_PATTERN.findall(response)
    allowed={c['id']:c['page'] for c,_ in hits}
    valid=bool(claims) and all(int(fid) in allowed and allowed[int(fid)]==int(page)
                               for fid,page in claims)
    refusal=bool(REFUSE_PATTERN.search(response))
    # A supported refusal has no citations; factual content requires references.
    return {'links':claims,'reference_valid':valid,'is_refusal':refusal,
            'citation_check_pass':valid or refusal}

def grounded_answer(query,hits):
    answer_1=answer(query,hits)
    result=validate_references(answer_1,hits)
    if result['citation_check_pass']:
        return answer_1,result,False
    allowed=', '.join(f"[Фрагмент {c['id']}, стр. {c['page']}]" for c,_ in hits)
    instruction=(f'Question: {query}\nAllowed citations ONLY: {allowed}.\n'
      'Re-answer based exclusively on the following PDF excerpts. Cite every factual claim. '
      'If insufficient, write exactly: Not enough information in the document.\n'
      +references(hits))
    answer_2=lm_generate(instruction)
    return answer_2,validate_references(answer_2,hits),True

answer_rows=[]
for item in QA[:5]:
    for method,searcher in searchers.items():
        hits=searcher(item[0]);out,audit,retried=grounded_answer(item[0],hits)
        answer_rows.append({'Метод':method,'Вопрос':item[0],'Эталон':item[1],
          'Ответ':out,'Цитаты':str(audit['links']),'Ссылки корректны':audit['reference_valid'],
          'Отказ':audit['is_refusal'],'Проверка формата пройдена':audit['citation_check_pass'],
          'Повторная генерация':retried,'ID top-3':[c['id'] for c,_ in hits],
          'Оценка фактов 0–2':'ПРОВЕРИТЬ'})
answer_df=pd.DataFrame(answer_rows)
display(answer_df[['Метод','Вопрос','Ссылки корректны','Отказ','Проверка формата пройдена','Повторная генерация']])

with pd.ExcelWriter('ЛР3_сравнение_RAG_улучшений.xlsx') as writer:
    improve_df.to_excel(writer,'Поиск_10_вопросов',index=False)
    answer_df.to_excel(writer,'Ответы_5_вопросов',index=False)
print('Saved: ЛР3_сравнение_RAG_улучшений.xlsx')

,Метод,Вопрос,Ссылки корректны,Отказ,Проверка формата пройдена,Повторная генерация
0,Dense,What is the title of the paper?,False,True,True,False
1,Hybrid (RRF),What is the title of the paper?,False,True,True,False
2,Hybrid + rerank,What is the title of the paper?,False,True,True,False
3,Dense,What name is given to the resulting models?,True,False,True,False
4,Hybrid (RRF),What name is given to the resulting models?,True,False,True,False
5,Hybrid + rerank,What name is given to the resulting models?,True,False,True,False
6,Dense,Which smaller InstructGPT model is preferred o...,True,False,True,False
7,Hybrid (RRF),Which smaller InstructGPT model is preferred o...,True,False,True,False
8,Hybrid + rerank,Which smaller InstructGPT model is preferred o...,True,False,True,False
9,Dense,What are the three main training steps?,True,False,True,False


Saved: ЛР3_сравнение_RAG_улучшений.xlsx


C:\Users\Admin\AppData\Local\Temp\ipykernel_16716\2525448692.py:40: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  improve_df.to_excel(writer,'Поиск_10_вопросов',index=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_16716\2525448692.py:41: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  answer_df.to_excel(writer,'Ответы_5_вопросов',index=False)


### Как оформить вывод

1. Укажите фактическую реализацию reranker (`RERANK_STATUS`). Если скачивание CrossEncoder не удалось, не называйте fallback нейросетевым reranking.
2. Сравните `Авто-якорь`, затем проверьте столбец `Текст top-3` вручную, поскольку одинаковая страница не всегда содержит достаточный ответ.
3. Сравните долю корректных ссылок и качество ответа 0–2 по пяти вопросам. **Не приравнивайте корректный ID к доказанности факта.**
4. Зафиксируйте отсутствие улучшения, если именно оно наблюдается.
5. Отчёт по обязательным семи заданиям не переписывайте под результаты этого дополнительного опыта; его вывод можно добавить отдельным подразделом.